# Day 01 — Exercises
**Complete these before moving to Day 02.**  
Solutions are in the final cell — but attempt each exercise first!

---


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("day01_exercises")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Ready")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 19:38:47 WARN Utils: Your hostname, jitendrarawatbitsinglasscoms-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.27 instead (on interface en0)
26/06/11 19:38:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 19:38:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


---
### Exercise 1 — Lazy evaluation chain
Create a DataFrame of integers 1–50.  
Filter to keep only numbers divisible by 3.  
Add a column `cubed` = the cube of each number.  
**Print the count first, then show the result.**  
Expected: 16 rows (3, 6, 9, … 48).


In [4]:
# Your code here
from pyspark.sql import functions as F
df = spark.range(1,51)\
     .filter(F.col("id") % 3 == 0)\
     .withColumn("cubed", (F.col("id") ** 3).cast("int"))

print(f"Count of dataframe: {df.count()}")
df.show()


Count of dataframe: 16
+---+------+
| id| cubed|
+---+------+
|  3|    27|
|  6|   216|
|  9|   729|
| 12|  1728|
| 15|  3375|
| 18|  5832|
| 21|  9261|
| 24| 13824|
| 27| 19683|
| 30| 27000|
| 33| 35937|
| 36| 46656|
| 39| 59319|
| 42| 74088|
| 45| 91125|
| 48|110592|
+---+------+



---
### Exercise 2 — GroupBy + aggregation
Using the employees data below, compute per-city: average salary, max salary, and headcount.  
Order by average salary descending.


In [9]:
employees = spark.createDataFrame([
    ("Alice",   "Toronto",  95_000),
    ("Bob",     "Montreal", 72_000),
    ("Carol",   "Toronto",  88_000),
    ("Dave",    "Calgary",  81_000),
    ("Eve",     "Montreal", 102_000),
    ("Frank",   "Calgary",  76_000),
    ("Grace",   "Toronto",  91_000),
], schema=["name", "city", "salary"])

# Your code here
expected_df = employees.groupBy("city")\
                .agg(
                    F.round(F.avg("salary"), 2).alias("average_salary"),
                    F.round(F.max("salary"), 2).alias("max_salary"),
                    F.count("*").alias("headcount")
                )\
                .orderBy("average_salary", ascending = False)


expected_df.show()

+--------+--------------+----------+---------+
|    city|average_salary|max_salary|headcount|
+--------+--------------+----------+---------+
| Toronto|      91333.33|     95000|        3|
|Montreal|       87000.0|    102000|        2|
| Calgary|       78500.0|     81000|        2|
+--------+--------------+----------+---------+



---
### Exercise 3 — Partitions
Using the employees DataFrame above:
1. Print its current partition count
2. Repartition to 4 partitions and print the count
3. Coalesce to 2 and print the count
4. In a comment: which of these caused a shuffle and why?


In [13]:
# Your code here
print(f"Current partition count: {employees.rdd.getNumPartitions()}")

employees_part4 = employees.repartition(4)
print(f"Modified partition count: {employees_part4.rdd.getNumPartitions()}")

employees_coalesced = employees.coalesce(2)
print(f"Coalesced partition count: {employees_coalesced.rdd.getNumPartitions()}")

print("Repartition of DF caused shuffle because it creates balanced partitions")


Current partition count: 15
Modified partition count: 4
Coalesced partition count: 2
Repartition of DF caused shuffle because it creates balanced partitions


---
### Exercise 4 — Conceptual (answer in comments)
Given this code:
```python
df = spark.range(1_000_000)
df2 = df.filter(df.id > 500_000)
df3 = df2.withColumn("doubled", df2.id * 2)
df4 = df3.groupBy((df3.doubled % 10).alias("last_digit")).count()
df4.show()
```
Answer in comments below:
- Q1: Which line(s) actually trigger computation?
- Q2: Classify each transformation as narrow or wide: filter, withColumn, groupBy, count
- Q3: How many stage boundaries (shuffles) do you expect? How many stages total?


In [ ]:
# Q1: df4.show()
# Q2: filter= narrow, withColumn=v narrow, groupBy= wide, count= wide
# Q3: stage boundaries= 1, total stages= 2


---
### Exercise 5 — Explicit schema
Define an explicit `StructType` schema for this data and create a DataFrame with it.  
Then print the schema and show the result.
```
product_id | product_name    | price  | in_stock
1          | Spark Book      | 49.99  | True
2          | Kafka Guide     | 44.99  | True  
3          | Flink Handbook  | 39.99  | False
```


In [15]:
# Your code here
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("in_stock", BooleanType(), False)
])

df = spark.createDataFrame([
    (1, "Spark Book",  49.99, True),
    (2, "Kafka Guide",  44.99, True),
    (3, "Flink Handbook",  39.99, False)
], schema = schema)

print(df.printSchema())

root
 |-- product_id: integer (nullable = false)
 |-- product_name: string (nullable = false)
 |-- price: double (nullable = false)
 |-- in_stock: boolean (nullable = false)

None


26/06/11 20:52:01 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

---
## Solutions (expand after attempting)


In [ ]:
# ── SOLUTION 1 ────────────────────────────────────────────────────────────────
divisible_by_3 = spark.range(1, 51).filter(F.col("id") % 3 == 0)
with_cubed = divisible_by_3.withColumn("cubed", F.col("id") ** 3)
print(f"Count: {with_cubed.count()}")
with_cubed.show()

# ── SOLUTION 2 ────────────────────────────────────────────────────────────────
(employees
 .groupBy("city")
 .agg(
     F.round(F.avg("salary"), 2).alias("avg_salary"),
     F.max("salary").alias("max_salary"),
     F.count("*").alias("headcount")
 )
 .orderBy("avg_salary", ascending=False)
 .show())

# ── SOLUTION 3 ────────────────────────────────────────────────────────────────
print(f"Original:       {employees.rdd.getNumPartitions()} partitions")
r4 = employees.repartition(4)
print(f"repartition(4): {r4.rdd.getNumPartitions()} partitions  ← full shuffle (can go up or down)")
c2 = r4.coalesce(2)
print(f"coalesce(2):    {c2.rdd.getNumPartitions()} partitions  ← no shuffle (merge local only, can only go down)")

# ── SOLUTION 4 ────────────────────────────────────────────────────────────────
# Q1: Only df4.show() triggers computation (it is the action)
# Q2: filter=narrow, withColumn=narrow, groupBy=wide, count=action(wide)
# Q3: 1 shuffle (at groupBy), 2 stages total (before shuffle, after shuffle)

# ── SOLUTION 5 ────────────────────────────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

product_schema = StructType([
    StructField("product_id",   IntegerType(), nullable=False),
    StructField("product_name", StringType(),  nullable=False),
    StructField("price",        DoubleType(),  nullable=False),
    StructField("in_stock",     BooleanType(), nullable=False),
])

products = spark.createDataFrame([
    (1, "Spark Book",     49.99, True),
    (2, "Kafka Guide",    44.99, True),
    (3, "Flink Handbook", 39.99, False),
], schema=product_schema)

products.printSchema()
products.show()


In [ ]:
spark.stop()
